In [1]:
import tensorflow as tf

# 1. Disable XLA execution and Auto-JIT optimization programmatically
tf.config.optimizer.set_jit(False)

# 2. Configure immediate eager execution metrics
tf.config.experimental_run_functions_eagerly(True) # Enforces step-by-step execution path
tf.data.experimental.enable_debug_mode()           # Disables heavy multi-threaded memory pinning

# 3. Dynamic VRAM Allocation Management
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            # Force TensorFlow to allocate memory incrementally rather than hogging the whole VRAM block
            tf.config.experimental.set_memory_growth(gpu, True)
        print("GPU Memory Growth Enabled successfully.")
    except RuntimeError as e:
        print(f"Memory growth configuration error: {e}")

I0000 00:00:1779544616.120524   20473 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1779544616.157935   20473 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1779544617.070940   20473 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


Instructions for updating:
Use `tf.config.run_functions_eagerly` instead of the experimental version.
GPU Memory Growth Enabled successfully.


In [4]:
"""
=============================================================================
Breast Cancer Detection — Mammogram Module (Local Machine)
Model   : EfficientNetB0 (Transfer Learning)
Dataset : CBIS-DDSM (Kaggle: awsaf49/cbis-ddsm-breast-cancer-image-dataset)
Output  : INT8 TFLite quantized model
Author  : Mark Sthembiso Mando | Mulungushi University MSc Data Science
=============================================================================
"""
import os
# Force stable asynchronous CUDA memory management
os.environ["TF_GPU_ALLOCATOR"] = "cuda_malloc_async"
# Completely disable XLA Auto-JIT Compilation to bypass dot_merger crashes
os.environ["TF_XLA_FLAGS"] = "--tf_xla_auto_jit=-1"
# Turn off aggressive oneDNN optimizations that conflict with pinning allocations
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"
import re
import glob
import random
import hashlib
import warnings
import time
import numpy as np
import pandas as pd
import cv2
import matplotlib
matplotlib.use("Agg")   # non-interactive backend — safe for scripts
import matplotlib.pyplot as plt
from tqdm import tqdm
 
import tensorflow as tf
from tensorflow.keras import layers, Model, callbacks, mixed_precision
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, roc_auc_score, average_precision_score,
    roc_curve, precision_recall_curve,
    confusion_matrix, ConfusionMatrixDisplay,
)
from sklearn.utils.class_weight import compute_class_weight
 
warnings.filterwarnings("ignore")

# =============================================================================
# 1. CONFIGURATION  ← Only edit this section
# =============================================================================
 
if os.name == "nt":                              # Native Windows
    DATASET_ROOT = r"C:\Users\mmando\Downloads\CBIS-DDSM"
    OUTPUT_DIR   = r"C:\Users\mmando\Documents\School\ML\Output_efficient"
else:                                            # WSL2 / Linux
    DATASET_ROOT = os.path.expanduser("~/CBIS-DDSM")
    OUTPUT_DIR   = os.path.expanduser("~/mammogram_output_Efficient")
 
# ── Training hyperparameters (proposal Section 4.4.2) ────────────────────────
IMG_SIZE        = 224       # EfficientNetB0 standard input size
BATCH_SIZE      = 16
EPOCHS_FROZEN   = 100       # Phase 1: frozen convolutional base
EPOCHS_FINETUNE = 30        # Phase 2: unfreeze last 15 layers
LEARNING_RATE   = 0.0001    # Adam initial LR
DROPOUT_RATE    = 0.3
L2_DECAY        = 0.0001
PATIENCE_STOP   = 15        # EarlyStopping patience
PATIENCE_LR     = 10        # ReduceLROnPlateau patience
SEED            = 42
 
# ── Data split (proposal Section 4.3) ────────────────────────────────────────
TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
TEST_RATIO  = 0.15
 
# ── TFLite (proposal Section 4.5.1) ──────────────────────────────────────────
TFLITE_CALIBRATION_SAMPLES = 1000
 
# =============================================================================
# Derived paths & setup — do not edit below this line
# =============================================================================
 
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
 
os.makedirs(OUTPUT_DIR, exist_ok=True)
CACHE_DIR          = os.path.join(OUTPUT_DIR, "cache")
CHECKPOINT_PATH_P1 = os.path.join(OUTPUT_DIR, "best_efficientnetb0_phase1.keras")
CHECKPOINT_PATH_P2 = os.path.join(OUTPUT_DIR, "best_efficientnetb0_phase2.keras")
CHECKPOINT_PATH    = CHECKPOINT_PATH_P2   
TFLITE_PATH        = os.path.join(OUTPUT_DIR, "mammogram_efficientnetb0_int8.tflite")
PLOT_DIR           = os.path.join(OUTPUT_DIR, "plots")
os.makedirs(PLOT_DIR,  exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)
 
print(f"\n{'='*62}")
print("  CBIS-DDSM  |  EfficientNetB0  |  INT8 TFLite")
print(f"{'='*62}\n")
print(f"  Platform   : {'Windows' if os.name == 'nt' else 'WSL2/Linux'}")
print(f"  TensorFlow : {tf.__version__}")
gpus = tf.config.list_physical_devices("GPU")
print(f"  GPUs found : {len(gpus)}")
if gpus:
    for g in gpus:
        tf.config.experimental.set_memory_growth(g, True)
    print(f"  Using      : {gpus[0].name}")
    print(f"  Precision  : mixed_float16 (Tensor Core acceleration)")
else:
    print("  WARNING    : No GPU detected — training will be slow.")
print(f"  Dataset    : {DATASET_ROOT}")
print(f"  Output     : {OUTPUT_DIR}")
print()

# =============================================================================
# 2. DATA LOADING
# =============================================================================
def load_dataset() -> pd.DataFrame:
    dicom_info_path = os.path.join(DATASET_ROOT, "csv", "dicom_info.csv")
    if not os.path.exists(dicom_info_path):
        matches = glob.glob(os.path.join(DATASET_ROOT, "**", "dicom_info.csv"), recursive=True)
        if not matches:
            raise FileNotFoundError(f"dicom_info.csv not found inside {DATASET_ROOT}.")
        dicom_info_path = matches[0]
 
    dicom_info = pd.read_csv(dicom_info_path)
    
    def resolve_image_path(raw: str) -> str:
        raw = re.sub(r"^CBIS-DDSM[/\\\\]", "", str(raw).strip())
        candidate = os.path.join(DATASET_ROOT, raw)
        return candidate if os.path.exists(candidate) else None
 
    dicom_info["full_path"] = dicom_info["image_path"].apply(resolve_image_path)
    series_to_path = dicom_info[dicom_info["full_path"].notna()].set_index("SeriesInstanceUID")["full_path"].to_dict()
    
    target_names = ["mass_case_description_train_set", "mass_case_description_test_set", "calc_case_description_train_set", "calc_case_description_test_set"]
    all_csvs  = glob.glob(os.path.join(DATASET_ROOT, "**", "*.csv"), recursive=True)
    csv_paths = [p for p in all_csvs if any(t in os.path.basename(p) for t in target_names)]
    
    dfs = [pd.read_csv(p) for p in csv_paths]
    df  = pd.concat(dfs, ignore_index=True)
    
    path_col = next((c for c in df.columns if "pathology" in c.lower()), None)
    path_cols = [c for c in df.columns if "file path" in c.lower() or ("path" in c.lower() and "image" in c.lower())]
 
    def extract_series_uid(dcm_str: str) -> str:
        parts = re.split(r"[/\\\\]", str(dcm_str).strip())
        return parts[-2] if len(parts) >= 2 else None
 
    best_col, best_df, best_count = None, None, 0
    for col in path_cols:
        tmp = df[[col, path_col]].dropna().copy()
        tmp.columns = ["dcm_path", "pathology"]
        tmp["series_uid"] = tmp["dcm_path"].apply(extract_series_uid)
        tmp["img_path"]   = tmp["series_uid"].map(series_to_path)
        n = tmp["img_path"].notna().sum()
        if n > best_count:
            best_col, best_df, best_count = col, tmp, n
 
    print(f'\nUsing: "{best_col}" ({best_count:,} images)')
 
    result = best_df.dropna(subset=["img_path"]).copy()
    result["label"] = result["pathology"].apply(lambda x: 1 if str(x).strip().upper() == "MALIGNANT" else 0)
    result = result.reset_index(drop=True)
 
    print(f"\n  Total     : {len(result):,}")
    print(f"  Benign    : {(result.label==0).sum():,}  ({(result.label==0).mean()*100:.1f}%)")
    print(f"  Malignant : {(result.label==1).sum():,}  ({(result.label==1).mean()*100:.1f}%)\n")
 
    return result

# =============================================================================
# 3. PREPROCESSING & CACHING
# =============================================================================
def apply_clahe(img_bgr: np.ndarray) -> np.ndarray:
    lab          = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
    clahe        = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    lab[:, :, 0] = clahe.apply(lab[:, :, 0])
    return cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)
 
def _cache_path(img_path: str) -> str:
    key = hashlib.md5(img_path.encode()).hexdigest()
    return os.path.join(CACHE_DIR, f"{key}.npy")
 
def build_cache(paths: list) -> None:
    missing = [p for p in paths if not os.path.exists(_cache_path(p))]
    if not missing:
        print(f"  Cache up to date — {len(paths):,} files already cached. ✅")
        return
    print(f"  Caching {len(missing):,} images → {CACHE_DIR}")
    for p in tqdm(missing, desc="  Building cache"):
        img = cv2.imread(p)
        if img is None:
            arr = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.float32)
        else:
            img = apply_clahe(img)
            img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            arr = img.astype(np.float32) / 255.0
        np.save(_cache_path(p), arr)
    print(f"  Cache complete — {len(paths):,} files. ✅\n")
 
def load_and_preprocess(img_path: str, augment: bool = False) -> np.ndarray:
    cp = _cache_path(img_path)
    if os.path.exists(cp):
        img_float = np.load(cp)
        img       = (img_float * 255).astype(np.uint8)
        img       = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
    else:
        img = cv2.imread(img_path)
        if img is None:
            return np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.float32)
        img = apply_clahe(img)
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
 
    if augment:
        if random.random() < 0.5:
            img = cv2.flip(img, 1)
        angle = random.uniform(-15, 15)
        M     = cv2.getRotationMatrix2D((IMG_SIZE // 2, IMG_SIZE // 2), angle, 1)
        img   = cv2.warpAffine(img, M, (IMG_SIZE, IMG_SIZE))
        hsv          = cv2.cvtColor(img, cv2.COLOR_BGR2HSV).astype(np.float32)
        hsv[:, :, 2] = np.clip(hsv[:, :, 2] * random.uniform(0.8, 1.2), 0, 255)
        img          = cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2BGR)
        zoom   = random.uniform(0.9, 1.1)
        new_sz = int(IMG_SIZE * zoom)
        img    = cv2.resize(img, (new_sz, new_sz))
        if zoom > 1.0:
            s   = (new_sz - IMG_SIZE) // 2
            img = img[s:s + IMG_SIZE, s:s + IMG_SIZE]
        else:
            p   = (IMG_SIZE - new_sz) // 2
            img = cv2.copyMakeBorder(img, p, p, p, p, cv2.BORDER_REFLECT)
            img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
        noise = np.random.normal(0, 0.01 * 255, img.shape).astype(np.float32)
        img   = np.clip(img.astype(np.float32) + noise, 0, 255).astype(np.uint8)
 
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return img.astype(np.float32) / 255.0

# =============================================================================
# 4. tf.data PIPELINE (Eager-Safe and Graph-Resilient String Handling)
# =============================================================================
def make_tf_dataset(paths: list, labels: list, augment: bool = False, shuffle: bool = False) -> tf.data.Dataset:
    def _load(path, label):
        def _parse_path(p):
            # If TensorFlow hands it over as a NumPy array due to eager mode execution
            if hasattr(p, "item"):
                p = p.item()
            # If it is still packed as raw bytes, decode it to a clean UTF-8 string
            if isinstance(p, bytes):
                return p.decode("utf-8")
            return str(p)

        img = tf.numpy_function(
            func=lambda p: load_and_preprocess(_parse_path(p), augment=augment),
            inp=[path], Tout=tf.float32,
        )
        img.set_shape([IMG_SIZE, IMG_SIZE, 3])
        return img, label
 
    ds = tf.data.Dataset.from_tensor_slices((paths, np.array(labels, dtype=np.float32)))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(paths), seed=SEED)
    
    ds = ds.map(_load, num_parallel_calls=4)
    if not augment:
        ds = ds.cache()
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

# =============================================================================
# 5. MODEL — EfficientNetB0 Architecture Integration
# =============================================================================
def build_model(freeze_base: bool = True):
    base = EfficientNetB0(
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
        include_top=False,
        weights="imagenet",
    )
    base.trainable = not freeze_base
 
    reg = tf.keras.regularizers.l2(L2_DECAY)
    inp = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    
    # Rescale inputs back from [0, 1] to [0, 255] as internal scaling layer handles the rest
    x = layers.Lambda(lambda v: v * 255.0, name="efficientnet_scaling")(inp)
    
    x   = base(x, training=not freeze_base)
    x   = layers.GlobalAveragePooling2D()(x)
    x   = layers.Dropout(DROPOUT_RATE)(x)
    x   = layers.Dense(256, activation="relu", kernel_regularizer=reg)(x)
    x   = layers.Dropout(DROPOUT_RATE)(x)
    out = layers.Dense(1, activation="sigmoid")(x)
 
    return Model(inp, out, name="EfficientNetB0_Mammogram"), base
 
def compile_model(model: Model) -> None:
    model.compile(
        optimizer=Adam(learning_rate=LEARNING_RATE, beta_1=0.9, beta_2=0.999),
        loss="binary_crossentropy",
        metrics=[
            "accuracy",
            tf.keras.metrics.AUC(name="auc"),             
            tf.keras.metrics.AUC(curve="PR", name="prc"), 
            tf.keras.metrics.Precision(name="precision"),
            tf.keras.metrics.Recall(name="recall"),
        ],
    )

# =============================================================================
# 6. CALLBACKS
# =============================================================================
def get_callbacks(checkpoint_path: str) -> list:
    return [
        callbacks.ModelCheckpoint(checkpoint_path, monitor="val_auc", mode="max", save_best_only=True, verbose=1),
        callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=PATIENCE_STOP, restore_best_weights=True, verbose=1),
        callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=PATIENCE_LR, min_lr=1e-7, verbose=1),
        callbacks.TensorBoard(log_dir=os.path.join(OUTPUT_DIR, "logs"), histogram_freq=1),
    ]

# =============================================================================
# 7. TRAINING
# =============================================================================
def train(train_ds, val_ds, cw_dict: dict, skip_phase1: bool = False):
    model, base = build_model(freeze_base=True)
    compile_model(model)
 
    if skip_phase1:
        print("\n─── Phase 1: Skipped ───")
        model.load_weights(CHECKPOINT_PATH_P1)
        h_frozen = None
    else:
        print("\n─── Phase 1: Frozen base ───")
        model.summary(line_length=80)
        h_frozen = model.fit(
            train_ds, validation_data=val_ds, epochs=EPOCHS_FROZEN,
            class_weight=cw_dict, callbacks=get_callbacks(CHECKPOINT_PATH_P1), verbose=1,
        )
        model.load_weights(CHECKPOINT_PATH_P1)
 
    print("\n─── Phase 2: Fine-tuning last 15 layers ───")
    base.trainable = True
    
    for layer in base.layers[:-15]:
        layer.trainable = False
        
    trainable = sum(1 for l in base.layers if l.trainable)
    print(f"  Trainable base layers : {trainable}")
 
    model.compile(
        optimizer=Adam(learning_rate=LEARNING_RATE * 0.1, beta_1=0.9, beta_2=0.999),
        loss="binary_crossentropy",
        metrics=["accuracy", tf.keras.metrics.AUC(name="auc"), tf.keras.metrics.AUC(curve="PR", name="prc"), tf.keras.metrics.Precision(name="precision"), tf.keras.metrics.Recall(name="recall")],
    )
 
    h_finetune = model.fit(
        train_ds, validation_data=val_ds, epochs=EPOCHS_FINETUNE,
        class_weight=cw_dict, callbacks=get_callbacks(CHECKPOINT_PATH_P2), verbose=1,
    )
    model.load_weights(CHECKPOINT_PATH_P2)
 
    return model, base, h_frozen, h_finetune


  CBIS-DDSM  |  EfficientNetB0  |  INT8 TFLite

  Platform   : WSL2/Linux
  TensorFlow : 2.21.0
  GPUs found : 1
  Using      : /physical_device:GPU:0
  Precision  : mixed_float16 (Tensor Core acceleration)
  Dataset    : /home/mmando/CBIS-DDSM
  Output     : /home/mmando/mammogram_output_Efficient



In [5]:
# =============================================================================
# 8. TRAINING HISTORY PLOT
# =============================================================================
def plot_history(h1, h2) -> None:
    metrics = [("accuracy", "Accuracy"), ("loss", "Loss"), ("auc", "ROC-AUC"), ("prc", "AUPRC")]
    phase1_skipped = h1 is None
    fig, axes = plt.subplots(1, 4, figsize=(20, 4))
 
    for ax, (key, title) in zip(axes, metrics):
        if phase1_skipped:
            train_vals = h2.history[key]
            val_vals   = h2.history[f"val_{key}"]
            ax.plot(train_vals, label="Train (Phase 2)")
            ax.plot(val_vals,   label="Validation (Phase 2)")
        else:
            train_vals = h1.history[key] + h2.history[key]
            val_vals   = h1.history[f"val_{key}"] + h2.history[f"val_{key}"]
            split      = len(h1.history["loss"]) - 1
            ax.plot(train_vals, label="Train")
            ax.plot(val_vals,   label="Validation")
            ax.axvline(split, color="gray", linestyle="--", alpha=0.7, label="Fine-tune start")
        ax.set_title(title, fontsize=12)
        ax.set_xlabel("Epoch")
        ax.legend(fontsize=8)
 
    title_str = "EfficientNetB0 — Training History (Phase 2 only)" if phase1_skipped else "EfficientNetB0 — Training History (Phase 1 + 2)"
    plt.suptitle(title_str, fontsize=14)
    plt.tight_layout()
    save_path = os.path.join(PLOT_DIR, "training_history.png")
    plt.savefig(save_path, dpi=150)
    plt.close()
    print(f"  Saved → {save_path}")

# =============================================================================
# 9. EVALUATION
# =============================================================================
def evaluate(model: Model, test_ds, y_te: list) -> dict:
    print("\n─── Test Set Evaluation ───")
    y_prob = model.predict(test_ds, verbose=1).ravel()
    y_pred = (y_prob >= 0.5).astype(int)
    y_true = np.array(y_te)
 
    roc_auc = roc_auc_score(y_true, y_prob)
    auprc   = average_precision_score(y_true, y_prob)
    report  = classification_report(y_true, y_pred, target_names=["Benign", "Malignant"], digits=4)
    print(f"\n  ROC-AUC : {roc_auc:.4f}  (H1 target ≥ 0.85)")
    print(f"  AUPRC   : {auprc:.4f}  (imbalance-aware supplement)")
    print(f"\n{report}")
 
    print("  Computing bootstrapped 95% CIs (1,000 iterations)...")
    rng = np.random.default_rng(SEED)
    boot_roc, boot_prc = [], []
    for _ in range(1000):
        idx = rng.choice(len(y_true), len(y_true), replace=True)
        if len(np.unique(y_true[idx])) < 2: 
            continue
        boot_roc.append(roc_auc_score(y_true[idx], y_prob[idx]))
        boot_prc.append(average_precision_score(y_true[idx], y_prob[idx]))
 
    roc_lo, roc_hi = np.percentile(boot_roc, [2.5, 97.5])
    prc_lo, prc_hi = np.percentile(boot_prc, [2.5, 97.5])
    print(f"  ROC-AUC 95% CI : [{roc_lo:.4f}, {roc_hi:.4f}]")
    print(f"  AUPRC   95% CI : [{prc_lo:.4f}, {prc_hi:.4f}]")
 
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
 
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    axes[0].plot(fpr, tpr, label=f"EfficientNetB0 (AUC={roc_auc:.3f})")
    axes[0].fill_between(fpr, tpr, alpha=0.1)
    axes[0].plot([0, 1], [0, 1], "k--")
    axes[0].set_xlabel("False Positive Rate")
    axes[0].set_ylabel("True Positive Rate")
    axes[0].set_title("ROC Curve")
    axes[0].legend()
 
    prec, rec, _ = precision_recall_curve(y_true, y_prob)
    axes[1].plot(rec, prec, label=f"EfficientNetB0 (AUPRC={auprc:.3f})")
    axes[1].fill_between(rec, prec, alpha=0.1)
    axes[1].set_xlabel("Recall")
    axes[1].set_ylabel("Precision")
    axes[1].set_title("Precision-Recall Curve")
    axes[1].legend()
 
    cm = confusion_matrix(y_true, y_pred)
    ConfusionMatrixDisplay(cm, display_labels=["Benign", "Malignant"]).plot(ax=axes[2], cmap="Blues")
    axes[2].set_title("Confusion Matrix")
 
    plt.suptitle("EfficientNetB0 — Mammogram Module Evaluation", fontsize=14)
    plt.tight_layout()
    save_path = os.path.join(PLOT_DIR, "evaluation.png")
    plt.savefig(save_path, dpi=150)
    plt.close()
    print(f"\n  Saved → {save_path}")
 
    return {
        "roc_auc": roc_auc, "roc_lo": roc_lo, "roc_hi": roc_hi,
        "auprc":   auprc,   "prc_lo": prc_lo, "prc_hi": prc_hi,
    }

# =============================================================================
# 10. GRAD-CAM 
# =============================================================================
def make_gradcam(img_array: np.ndarray, model: Model) -> np.ndarray:
    inp = tf.cast(img_array[np.newaxis], tf.float32)

    base_model = None
    pre_layers = []
    head_layers = []
    found_base = False

    for layer in model.layers:
        if isinstance(layer, tf.keras.Model):
            base_model = layer
            found_base = True
        elif not found_base:
            if not isinstance(layer, tf.keras.layers.InputLayer):
                pre_layers.append(layer)
        else:
            head_layers.append(layer)

    if base_model is None:
        raise ValueError("No nested Keras submodel found — cannot build Grad-CAM.")

    with tf.GradientTape() as tape:
        x = inp
        for layer in pre_layers:
            x = layer(x)
            
        conv_out = base_model(x, training=False)
        tape.watch(conv_out)

        x = conv_out
        for layer in head_layers:
            if isinstance(layer, tf.keras.layers.Dropout):
                x = layer(x, training=False)
            else:
                x = layer(x)

        preds = x
        class_score = tf.cast(preds[:, 0], tf.float32)

    grads = tape.gradient(class_score, conv_out)
    if grads is None:
        raise ValueError("Gradients are None — tape did not capture conv_out.")
        
    grads    = tf.cast(grads, tf.float32)
    conv_out = tf.cast(conv_out, tf.float32)
    
    pooled   = tf.reduce_mean(grads, axis=(0, 1, 2))
    heatmap  = tf.squeeze(conv_out[0] @ pooled[..., tf.newaxis])
    heatmap  = tf.maximum(heatmap, 0)
    heatmap  = heatmap / (tf.math.reduce_max(heatmap) + 1e-8)
    
    return heatmap.numpy()
 
def overlay_gradcam(img: np.ndarray, heatmap: np.ndarray, alpha: float = 0.4) -> np.ndarray:
    h = cv2.resize(heatmap, (img.shape[1], img.shape[0]))
    h = cv2.applyColorMap(np.uint8(255 * h), cv2.COLORMAP_JET)
    h = cv2.cvtColor(h, cv2.COLOR_BGR2RGB)
    return np.uint8(img * 255 * (1 - alpha) + h * alpha)
 
def save_gradcam_grid(model: Model, X_te: list, y_te: list) -> None:
    benign_idx    = [i for i, l in enumerate(y_te) if l == 0][:2]
    malignant_idx = [i for i, l in enumerate(y_te) if l == 1][:2]
    sample_idx    = benign_idx + malignant_idx
 
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    for col, idx in enumerate(sample_idx):
        img   = load_and_preprocess(X_te[idx], augment=False)
        prob  = float(model.predict(img[np.newaxis], verbose=0)[0, 0])
        label = "Malignant" if y_te[idx] == 1 else "Benign"
        tick  = "✓" if int(prob >= 0.5) == y_te[idx] else "✗"
        try:
            hm = make_gradcam(img, model)
            ov = overlay_gradcam(img, hm)
            axes[0, col].imshow(img)
            axes[0, col].set_title(f"Original\n({label})", fontsize=9)
            axes[0, col].axis("off")
            axes[1, col].imshow(ov)
            axes[1, col].set_title(f"Grad-CAM  {tick}\nP(malignant)={prob:.2f}", fontsize=9)
            axes[1, col].axis("off")
        except Exception as e:
            print(f"  Grad-CAM failed for sample {idx}: {e}")
 
    plt.suptitle("Grad-CAM — EfficientNetB0 Mammogram Module", fontsize=12)
    plt.tight_layout()
    save_path = os.path.join(PLOT_DIR, "gradcam.png")
    plt.savefig(save_path, dpi=150)
    plt.close()
    print(f"  Saved → {save_path}")

# =============================================================================
# 11. INT8 TFLITE QUANTIZATION
# =============================================================================
def quantize_to_tflite(model: Model, cal_paths: list, cal_labels: list) -> tuple:
    print("\n─── INT8 TFLite Quantization ───")
    paths  = cal_paths[:TFLITE_CALIBRATION_SAMPLES]
    labels = cal_labels[:TFLITE_CALIBRATION_SAMPLES]
 
    cal_imgs = np.array(
        [load_and_preprocess(p, augment=False) for p in tqdm(paths, desc="  Calibrating")],
        dtype=np.float32,
    )
 
    def representative_dataset():
        for img in cal_imgs:
            yield [img[np.newaxis]]
 
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    converter.optimizations          = [tf.lite.Optimize.DEFAULT]
    converter.representative_dataset = representative_dataset
    converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    converter.inference_input_type   = tf.uint8
    converter.inference_output_type  = tf.uint8
 
    try:
        tflite_model = converter.convert()
        quant_mode   = "INT8"
        print("  ✓ INT8 quantization successful")
    except Exception as e:
        print(f"  INT8 failed ({e}) — falling back to float16")
        conv2 = tf.lite.TFLiteConverter.from_keras_model(model)
        conv2.optimizations               = [tf.lite.Optimize.DEFAULT]
        conv2.target_spec.supported_types = [tf.float16]
        tflite_model = conv2.convert()
        quant_mode   = "float16"
 
    with open(TFLITE_PATH, "wb") as f:
        f.write(tflite_model)
 
    size_mb = os.path.getsize(TFLITE_PATH) / (1024 ** 2)
    print(f"\n  Mode   : {quant_mode}")
    print(f"  Size   : {size_mb:.2f} MB  (target ≤ 15 MB)")
 
    interpreter = tf.lite.Interpreter(
        model_path=TFLITE_PATH,
        experimental_op_resolver_type=tf.lite.experimental.OpResolverType.BUILTIN_WITHOUT_DEFAULT_DELEGATES,
    )
    interpreter.allocate_tensors()
    in_det   = interpreter.get_input_details()[0]
    out_det  = interpreter.get_output_details()[0]
    in_dtype = in_det["dtype"]
 
    test_img = load_and_preprocess(paths[0], augment=False)
    test_inp = (test_img * 255).astype(np.uint8)[np.newaxis] if in_dtype == np.uint8 else test_img[np.newaxis]
 
    run_times = []
    for _ in range(50):
        t0 = time.perf_counter()
        interpreter.set_tensor(in_det["index"], test_inp)
        interpreter.invoke()
        _ = interpreter.get_tensor(out_det["index"])
        run_times.append((time.perf_counter() - t0) * 1000)
 
    mean_ms = np.mean(run_times[5:])
    
    # Accuracy drop evaluation
    cal_ds    = make_tf_dataset(paths, labels)
    orig_prob = model.predict(cal_ds, verbose=0).ravel()
    orig_pred = (orig_prob >= 0.5).astype(int)
 
    interpreter2 = tf.lite.Interpreter(
        model_path=TFLITE_PATH,
        experimental_op_resolver_type=tf.lite.experimental.OpResolverType.BUILTIN_WITHOUT_DEFAULT_DELEGATES,
    )
    interpreter2.allocate_tensors()
    in_det2   = interpreter2.get_input_details()[0]
    out_det2  = interpreter2.get_output_details()[0]
    in_dtype2 = in_det2["dtype"]
 
    quant_pred = []
    for img in cal_imgs:
        inp = (img * 255).astype(np.uint8)[np.newaxis] if in_dtype2 == np.uint8 else img[np.newaxis]
        interpreter2.set_tensor(in_det2["index"], inp)
        interpreter2.invoke()
        out  = interpreter2.get_tensor(out_det2["index"])
        prob = out[0, 0] / 255.0 if in_dtype2 == np.uint8 else float(out[0, 0])
        quant_pred.append(int(prob >= 0.5))
 
    orig_acc  = np.mean(orig_pred == np.array(labels))
    quant_acc = np.mean(np.array(quant_pred) == np.array(labels))
    drop      = (orig_acc - quant_acc) * 100
    print(f"  Original accuracy  : {orig_acc:.4f}")
    print(f"  Quantized accuracy : {quant_acc:.4f}")
    print(f"  Drop               : {drop:.2f}%")
 
    return size_mb, mean_ms

# =============================================================================
# 12. MAIN PIPELINE
# =============================================================================
def main():
    print("─── Step 1 / 5 : Loading dataset ───")
    df     = load_dataset()
    paths  = df["img_path"].tolist()
    labels = df["label"].tolist()
 
    print("─── Step 2 / 5 : Splitting data ───")
    X_tr, X_tmp, y_tr, y_tmp = train_test_split(paths, labels, test_size=1 - TRAIN_RATIO, stratify=labels, random_state=SEED)
    val_frac = VAL_RATIO / (VAL_RATIO + TEST_RATIO)
    X_val, X_te, y_val, y_te = train_test_split(X_tmp, y_tmp, test_size=1 - val_frac, stratify=y_tmp, random_state=SEED)
    print(f"  Train : {len(X_tr):4d}  ({sum(y_tr)} malignant, {len(y_tr)-sum(y_tr)} benign)")
    print(f"  Val   : {len(X_val):4d}  ({sum(y_val)} malignant, {len(y_val)-sum(y_val)} benign)")
    print(f"  Test  : {len(X_te):4d}  ({sum(y_te)} malignant, {len(y_te)-sum(y_te)} benign)\n")
 
    cw      = compute_class_weight("balanced", classes=np.unique(y_tr), y=y_tr)
    cw_dict = {i: float(w) for i, w in enumerate(cw)}
 
    print("─── Step 2b: Building image cache ───")
    build_cache(paths)
 
    train_ds = make_tf_dataset(X_tr,  y_tr,  augment=True,  shuffle=True)
    val_ds   = make_tf_dataset(X_val, y_val, augment=False, shuffle=False)
    test_ds  = make_tf_dataset(X_te,  y_te,  augment=False, shuffle=False)
 
    print("─── Step 3 / 5 : Training ───")
    model, _, h_frozen, h_finetune = train(train_ds, val_ds, cw_dict, skip_phase1=False)
    plot_history(h_frozen, h_finetune)
 
    print("\n─── Step 4 / 5 : Evaluation ───")
    results = evaluate(model, test_ds, y_te)
 
    print("\n─── Grad-CAM visualisations ───")
    save_gradcam_grid(model, X_te, y_te)
 
    print("\n─── Step 5 / 5 : TFLite INT8 Quantization ───")
    size_mb, mean_ms = quantize_to_tflite(model, X_tr, y_tr)
 
    h1_auc     = results["roc_auc"] >= 0.85
    h1_size    = size_mb  <= 15
    h1_latency = mean_ms  <= 100
    h1_all     = h1_auc and h1_size and h1_latency
 
    print(f"\n{'='*62}")
    print("  PIPELINE COMPLETE")
    print(f"{'='*62}")
    print(f"  ROC-AUC  : {results['roc_auc']:.4f}  95% CI [{results['roc_lo']:.4f}, {results['roc_hi']:.4f}]")
    print(f"  AUPRC    : {results['auprc']:.4f}  95% CI [{results['prc_lo']:.4f}, {results['prc_hi']:.4f}]")
    print(f"  Size     : {size_mb:.2f} MB")
    print(f"  Latency  : {mean_ms:.1f} ms")
    print(f"\n  Overall H1 verdict : {'✓ SUPPORTED' if h1_all else '✗ NOT FULLY MET'}")
    print(f"{'='*62}\n")
 
if __name__ == "__main__":
    main()

─── Step 1 / 5 : Loading dataset ───

Using: "image file path" (3,568 images)

  Total     : 3,568
  Benign    : 2,111  (59.2%)
  Malignant : 1,457  (40.8%)

─── Step 2 / 5 : Splitting data ───
  Train : 2497  (1020 malignant, 1477 benign)
  Val   :  535  (218 malignant, 317 benign)
  Test  :  536  (219 malignant, 317 benign)

─── Step 2b: Building image cache ───
  Cache up to date — 3,568 files already cached. ✅
─── Step 3 / 5 : Training ───

─── Phase 1: Frozen base ───


Model: "EfficientNetB0_Mammogram"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                      ┃ Output Shape             ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)        │ (None, 224, 224, 3)      │             0 │
├───────────────────────────────────┼──────────────────────────┼───────────────┤
│ efficientnet_scaling (Lambda)     │ (None, 224, 224, 3)      │             0 │
├───────────────────────────────────┼──────────────────────────┼───────────────┤
│ efficientnetb0 (Functional)       │ (None, 7, 7, 1280)       │     4,049,571 │
├───────────────────────────────────┼──────────────────────────┼───────────────┤
│ global_average_pooling2d_1        │ (None, 1280)             │             0 │
│ (GlobalAveragePooling2D)          │                          │               │
├───────────────────────────────────┼──────────────────────────┼───────────────┤
│ dropout_2 (Dropout)               │ (None, 1280)             │             0 │
├───────────────────────────────────┼──────────────────────────┼───────────────┤
│ dense_2 (Dense)                   │ (None, 256)              │       327,936 │
├───────────────────────────────────┼──────────────────────────┼───────────────┤
│ dropout_3 (Dropout)               │ (None, 256)              │             0 │
├───────────────────────────────────┼──────────────────────────┼───────────────┤
│ dense_3 (Dense)                   │ (None, 1)                │           257 │
└───────────────────────────────────┴──────────────────────────┴───────────────┘

 Total params: 4,377,764 (16.70 MB)

 Trainable params: 328,193 (1.25 MB)

...

INFO:tensorflow:Assets written to: /tmp/tmptjnozo8h/assets


INFO:tensorflow:Assets written to: /tmp/tmptjnozo8h/assets


Saved artifact at '/tmp/tmptjnozo8h'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='keras_tensor_486')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  125841173383312: TensorSpec(shape=(1, 1, 1, 3), dtype=tf.float32, name=None)
  125841173383504: TensorSpec(shape=(1, 1, 1, 3), dtype=tf.float32, name=None)
  125838853009296: TensorSpec(shape=(), dtype=tf.resource, name=None)
  125838853009104: TensorSpec(shape=(), dtype=tf.resource, name=None)
  125838853008144: TensorSpec(shape=(), dtype=tf.resource, name=None)
  125838853010448: TensorSpec(shape=(), dtype=tf.resource, name=None)
  125838852997968: TensorSpec(shape=(), dtype=tf.resource, name=None)
  125838853009872: TensorSpec(shape=(), dtype=tf.resource, name=None)
  125838853011024: TensorSpec(shape=(), dtype=tf.resource, name=None)
  125838852999504: TensorSpec(shape=(), dtype=tf.resource, name=

W0000 00:00:1779569556.055193   20473 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1779569556.055658   20473 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
I0000 00:00:1779569556.064573   20473 reader.cc:83] Reading SavedModel from: /tmp/tmptjnozo8h
I0000 00:00:1779569556.077258   20473 reader.cc:52] Reading meta graph with tags { serve }
I0000 00:00:1779569556.077282   20473 reader.cc:147] Reading SavedModel debug info (if present) from: /tmp/tmptjnozo8h
I0000 00:00:1779569556.157268   20473 mlir_graph_optimization_pass.cc:437] MLIR V1 optimization pass is not enabled
I0000 00:00:1779569556.179499   20473 loader.cc:236] Restoring SavedModel bundle.
I0000 00:00:1779569556.498957   20473 loader.cc:220] Running initialization op on SavedModel bundle at path: /tmp/tmptjnozo8h
I0000 00:00:1779569556.582910   20473 loader.cc:471] SavedModel load for tags { serve }; Status: success: OK. Took 518367 microseconds.
I0000 00:00:1779569556.818734   2047

  ✓ INT8 quantization successful

  Mode   : INT8
  Size   : 5.03 MB  (target ≤ 15 MB)
  Original accuracy  : 0.7040
  Quantized accuracy : 0.5690
  Drop               : 13.50%

  PIPELINE COMPLETE
  ROC-AUC  : 0.7121  95% CI [0.6676, 0.7561]
  AUPRC    : 0.5805  95% CI [0.5181, 0.6531]
  Size     : 5.03 MB
  Latency  : 36.8 ms

  Overall H1 verdict : ✗ NOT FULLY MET

